# 02 · GGUF + imatrix: quality and CPU speed
**Runtime:** T4 GPU (quality) + this machine's CPU (speed). If the CUDA build fails, everything runs on CPU
(WikiText ppl on a ~41k-token subset; KL divergence unchanged). **Writes:** `results_02_gguf.jsonl`. **Rough time:** ~1.5–3 h.

| config_id | What |
|---|---|
| `gguf_f16` | llama.cpp anchor (Δ% in this tool are vs this row) |
| `gguf_{q8_0, q5_k_m, q4_k_m, q4_0, iq4_xs, q3_k_m}` | No imatrix: quality + CPU speed vs the bandwidth ceiling |
| `..._imat_wiki` | imatrix from WikiText-2 **train** (eval on **test**) |
| `..._imat_code` | imatrix from code (Q4_K_M, IQ4_XS, Q3_K_M) |

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!pip install -q -U datasets huggingface_hub   # pip warnings are safe to ignore

## Setup
Set `PLATFORM` (`"colab"` or `"kaggle"`). The helper `edgelayer_bench.py` is fetched from the repo
(or used directly if you cloned the repo). Every finished config is saved to `RESULTS_FILE` right away,
so after a disconnect you just re-run all cells: finished configs are skipped.

In [ ]:
PLATFORM = "kaggle"                      # "colab" | "kaggle"
RESULTS_FILE = "results_02_gguf.jsonl"
REPO_RAW = "https://raw.githubusercontent.com/<your-username>/edgelayer-quantization/main"   # <- replace once

import os, sys, json, urllib.request
BENCH_DIR = {"colab": "/content/drive/MyDrive/edgelayer-bench",
             "kaggle": "/kaggle/working/edgelayer-bench"}[PLATFORM]
if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(BENCH_DIR, exist_ok=True)

# Use the repo's helper if this notebook runs from a clone, else download it once
for d in ["..", "."]:
    if os.path.exists(f"{d}/edgelayer_bench.py"):
        sys.path.insert(0, os.path.abspath(d)); break
else:
    urllib.request.urlretrieve(f"{REPO_RAW}/edgelayer_bench.py", f"{BENCH_DIR}/edgelayer_bench.py")
    sys.path.insert(0, BENCH_DIR)

In [ ]:
import torch
import edgelayer_bench as eb
rd = eb.setup(BENCH_DIR, platform=PLATFORM, results_file=RESULTS_FILE)
MODEL_ID = eb.CFG["model_id"]
eb.ensure_meta()                            # parameter counts, used for bits/weight

def done_ids():
    """config_ids that already have a row with this field (default: any row)."""
    p = rd / RESULTS_FILE
    return {json.loads(l)["config_id"] for l in open(p) if l.strip()} if p.exists() else set()

print(MODEL_ID, "|", eb.ram_gb()); eb.env_info()

## Build llama.cpp (CPU + CUDA; skipped if already built)

In [ ]:
import glob, shutil
W = eb.CFG["work_dir"]
LC = f"{W}/llama.cpp"
BIN, CPU_BIN = f"{LC}/build-cuda/bin", f"{LC}/build-cpu/bin"
TOOLS = ["llama-bench", "llama-imatrix", "llama-perplexity", "llama-quantize"]
built = lambda b: all(os.path.exists(f"{b}/{t}") for t in TOOLS)

nvcc = shutil.which("nvcc") or next(iter(sorted(glob.glob("/usr/local/cuda*/bin/nvcc"))), None)
if nvcc:                                            # CMake must be able to find nvcc
    os.environ["PATH"] = os.path.dirname(nvcc) + ":" + os.environ["PATH"]; os.environ["CUDACXX"] = nvcc
if not built(CPU_BIN):
    eb.build_llamacpp(cuda=False)                   # ~5-10 min
GPU_OK = False
if nvcc:
    if not built(BIN):
        shutil.rmtree(f"{LC}/build-cuda", ignore_errors=True)
        try:
            eb.build_llamacpp(cuda=True, cuda_arch="75")
        except RuntimeError as e:
            print("CUDA build failed, quality runs fall back to CPU:", e)
    GPU_OK = built(BIN)
if not GPU_OK:
    BIN = CPU_BIN
NGL = "-ngl 99" if GPU_OK else ""
PPL_CHUNKS = "" if GPU_OK else "--chunks 20"         # full WikiText test on GPU; ~41k tokens on CPU
LLAMA_CPP = eb.sh(f"git -C {LC} log -1 --format='%h %cd'").strip()
print("llama.cpp", LLAMA_CPP, "| GPU quality runs:", GPU_OK)
!pip install -q {LC}/gguf-py
P, N, R = eb.CFG["speed_prompt"], eb.CFG["speed_gen"], eb.CFG["speed_reps"]

## GGUF F16, eval/calibration text, imatrix files, FP16 logits for KL divergence

In [ ]:
from huggingface_hub import snapshot_download
hf_dir = snapshot_download(MODEL_ID, local_dir=f"{W}/hf_model")
G = rd / "gguf"; G.mkdir(exist_ok=True)
F16 = G / "model-f16.gguf"
if not F16.exists():
    eb.sh(f"python {LC}/convert_hf_to_gguf.py {hf_dir} --outtype f16 --outfile {F16}")

def write_once(path, make):
    if not os.path.exists(path): eb.write_text_file(make(), path)
    return path

WIKI_TEST = write_once(f"{W}/wiki.test.raw", lambda: eb.wikitext_text("test"))
CODE_TEST = write_once(f"{W}/code.test.txt", eb.domain_text)
CAL = {"wiki": write_once(f"{W}/calib.wiki.txt", lambda: eb.wikitext_text("train")),
       "code": write_once(f"{W}/calib.code.txt", lambda: "\n\n".join(eb.domain_texts(2000)))}

IMAT = {}
for name, f in CAL.items():
    IMAT[name] = f"{W}/imatrix-{name}.gguf"
    if not os.path.exists(IMAT[name]):
        eb.sh(f"{BIN}/llama-imatrix -m {F16} -f {f} -o {IMAT[name]} -c 512 --chunks 200 {NGL}")
KLD_BASE = f"{W}/f16.kld"                           # saved FP16 logits (a few GB, local disk)
if not os.path.exists(KLD_BASE):
    eb.sh(f"{BIN}/llama-perplexity -m {F16} -f {WIKI_TEST} -c 512 --chunks 40 --kl-divergence-base {KLD_BASE} {NGL}")
BW = eb.cpu_bandwidth_gbs()
print("CPU copy bandwidth ~", BW, "GB/s on", os.cpu_count(), "threads")

## Quantize and evaluate (finished configs are skipped)

In [ ]:
import subprocess

def bench(binary, path, extra):
    r = subprocess.run(f"{binary}/llama-bench -m {path} -p {P} -n {N} -r {R} {extra} -o json",
                       shell=True, capture_output=True, text=True)
    return eb.parse_llama_bench(r.stdout) if r.returncode == 0 else {}

def gguf_eval(path, speed=True):
    ppl, err = eb.parse_ppl(eb.sh(f"{BIN}/llama-perplexity -m {path} -f {WIKI_TEST} -c 2048 {PPL_CHUNKS} {NGL}"))
    ppl_code, _ = eb.parse_ppl(eb.sh(f"{BIN}/llama-perplexity -m {path} -f {CODE_TEST} -c 2048 "
                                     f"--chunks {eb.CFG['domain_max_windows']} {NGL}"))
    kld, top1 = eb.parse_kld(eb.sh(f"{BIN}/llama-perplexity -m {path} -f {WIKI_TEST} -c 512 --chunks 40 "
                                   f"--kl-divergence-base {KLD_BASE} --kl-divergence {NGL}"))
    size = eb.file_size_gb(path)
    row = dict(ppl_wiki_lcpp=ppl, ppl_wiki_lcpp_err=err, ppl_code_lcpp=ppl_code, kld_lcpp=kld, top1_lcpp=top1,
               size_gb=size, bpw=eb.bpw(size), llama_cpp=LLAMA_CPP, ppl_subset=not GPU_OK)
    if speed:                                       # imatrix changes quality, not speed
        if GPU_OK:
            row.update(**eb.prefixed(bench(BIN, path, NGL), eb.gpu_tag()))
        c = bench(CPU_BIN, path, f"-t {os.cpu_count()}")
        ceil = round(BW / size, 1)
        row.update(engine="llama.cpp", **eb.prefixed(c, "cpu"), cpu_bw_gbs=BW, cpu_ceiling_tok_s=ceil,
                   cpu_efficiency=round(c["decode_tok_s"] / ceil, 2) if c.get("decode_tok_s") else None)
    return row

if "gguf_f16" not in done_ids():
    eb.log_result("gguf_f16", post=7, method="GGUF F16", bits=16, **gguf_eval(F16))

QUANTS = ["Q8_0", "Q5_K_M", "Q4_K_M", "Q4_0", "IQ4_XS", "Q3_K_M"]
RUNS = ([(q, None) for q in QUANTS] + [(q, "wiki") for q in QUANTS if q != "Q8_0"]
        + [(q, "code") for q in ["Q4_K_M", "IQ4_XS", "Q3_K_M"]])
for q, imat in RUNS:
    cfg_id = f"gguf_{q.lower()}" + (f"_imat_{imat}" if imat else "")
    if cfg_id in done_ids(): print("skip", cfg_id); continue
    path = (G / f"model-{q}.gguf") if imat is None else f"{W}/model-{q}-{imat}.gguf"
    try:
        with eb.Timer() as t:
            if not os.path.exists(path):
                eb.sh(f"{BIN}/llama-quantize {'--imatrix ' + IMAT[imat] if imat else ''} {F16} {path} {q}")
        eb.log_result(cfg_id, post=7, method=f"GGUF {q}", quant=q, imatrix=imat, quant_time_s=t.s,
                      **gguf_eval(path, speed=imat is None))
    except Exception as e:
        print("FAILED", cfg_id, e)
    if imat and os.path.exists(path): os.remove(path)

## Copy the results
Downloads from Kaggle can fail: the cell below prints the file so you can copy it.

In [ ]:
print(open(rd / "results_02_gguf.jsonl").read())